# Character features & switch-target training


### Why
In the evaluation notebook we found that the BiLSTM's token-level
AUC of 0.99 was hiding a much weaker result on the actual linguistic task:

| Metric         | word+POS BiLSTM |
|----------------|----------------:|
| Token F1       | 0.989 |
| Switch F1      | 0.636 |
| Switch AP      | 0.724 |

The model misses ~40% of real switches. 
1. **Character-level features.** Spanish and English have very different
   spelling patterns (`ción`, `ñ`, `ll`, `qu`, `th`, `sh`, `-ing`, …).
   A small convolutional encoder over characters gives the model a strong
   signal about a word's language even when the word itself is rare or
   unseen.
2. **Train directly on the switch target** with a class-weighted loss, so
   that the model optimizes the metric we actually care about (rare-event
   detection) rather than per-token language, where 98 % of labels are
   trivial continuations.


Pretrained multilingual embeddings (fastText / XLM-R) would be the
next-next step 

## 1. Preprocessing




In [1]:
import os, time, random, math, json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score, confusion_matrix,
    average_precision_score, f1_score, accuracy_score,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CORPUS = Path("../Data/CS Corpus Prediction/BangorCorpus.txt")
assert CORPUS.exists(), CORPUS

In [2]:
# --- Load + tokenize to word-level rows (same as tutorial) ---
df = pd.read_csv(CORPUS, sep="\t")
TOP_LANGS = ("eng", "spa")

def tokenize(utt, syn):
    if not isinstance(utt, str) or not isinstance(syn, str): return None
    toks, tags = utt.split(","), syn.split(".")
    if len(toks) != len(tags): return None
    out = []
    for tok, pos in zip(toks, tags):
        if "." not in tok: return None
        w, l = tok.rsplit(".", 1)
        out.append((w.strip(), l.strip(), pos.strip()))
    return out
def ok(triples, allowed=("eng","spa","amb")):
    return all(l in allowed for _, l, _ in triples)

rows = []
dialogue_to_id = {sf: i for i, sf in enumerate(sorted(df["Soundfile"].unique()))}
sent_cnt = defaultdict(int)
for _, r in df.iterrows():
    t = tokenize(r["Utterance"], r["Syntax"])
    if t is None or not ok(t):
        continue
    did = dialogue_to_id[r["Soundfile"]]; sent_cnt[did] += 1; sid = sent_cnt[did]
    for wi, (w, l, p) in enumerate(t, 1):
        rows.append({"dialogue_id": did, "sentence_id": sid,
                     "word_index": wi, "word": w, "word_lang": l, "pos_tag": p})
tidy = pd.DataFrame(rows)

data = tidy[tidy["word_lang"].isin(TOP_LANGS)].copy().reset_index(drop=True)
data["y"] = (data["word_lang"] == "spa").astype(int)

rng = np.random.default_rng(SEED)
dials = np.array(sorted(data["dialogue_id"].unique())); rng.shuffle(dials)
n = len(dials); n_tr, n_va = int(0.8*n), int(0.1*n)
tr_d = set(dials[:n_tr]); va_d = set(dials[n_tr:n_tr+n_va])
def split_name(d):
    if d in tr_d: return "train"
    if d in va_d: return "val"
    return "test"
data["split"] = data["dialogue_id"].map(split_name)
data = data.sort_values(["dialogue_id","sentence_id","word_index"]).reset_index(drop=True)

grp = data.groupby(["dialogue_id","sentence_id"], sort=False)
data["prev_y"] = grp["y"].shift(1)
data["switch_true"] = (data["prev_y"].notna() &
                       (data["y"] != data["prev_y"])).astype("float")
print("Split sizes:", data.groupby("split").size().to_dict())
print("Switch rate (on tokens with a previous):",
      round(data.loc[data["prev_y"].notna(), "switch_true"].mean(), 4))

Split sizes: {'test': 37027, 'train': 182997, 'val': 28134}
Switch rate (on tokens with a previous): 0.0178


## 2. Building vocabularies — word, POS, and character embedding

**character vocabulary** — we collect every character that
appears often enough in training, lowercase everything, and reserve two
special slots (`<pad>` for padding, `<unk>` for rare chars).



In [3]:
PAD, UNK = "<pad>", "<unk>"
tr_words = data[data["split"]=="train"]

# word vocab
wc = Counter(tr_words["word"].tolist())
vocab = [PAD, UNK] + [w for w, c in wc.most_common() if c >= 2]
word2id = {w:i for i,w in enumerate(vocab)}

# POS vocab
pos_vocab = [PAD] + sorted(tr_words["pos_tag"].unique().tolist())
pos2id = {p:i for i,p in enumerate(pos_vocab)}

# char vocab (lowercase, keep chars seen >= 5 times)
char_counter = Counter()
for w in tr_words["word"]:
    for ch in w.lower():
        char_counter[ch] += 1
chars = [PAD, UNK] + [c for c, cnt in char_counter.most_common() if cnt >= 5]
char2id = {c:i for i,c in enumerate(chars)}
print(f"Word vocab {len(word2id):,}  |  POS vocab {len(pos2id)}  |  Char vocab {len(char2id)}")

enc_w = lambda w: word2id.get(w, word2id[UNK])
enc_p = lambda p: pos2id.get(p, 0)

# Each word gets encoded as a fixed-length sequence of char ids.  Words are
# padded/truncated to MAX_CHAR characters so they batch cleanly.
MAX_CHAR = 20
def enc_chars(w, max_c=MAX_CHAR):
    w = w.lower()[:max_c]
    ids = [char2id.get(c, char2id[UNK]) for c in w]
    ids = ids + [0] * (max_c - len(ids))
    return ids

Word vocab 5,599  |  POS vocab 48  |  Char vocab 38


## 3. Per-sentence tensors with three parallel label tracks

For every sentence we keep:

- `words_id` — list of word ids (as before)
- `pos` — list of POS ids (as before)
- `chars` — list of `(MAX_CHAR,)` char-id vectors, one per word (new)
- `y` — per-token language label, 0 = eng, 1 = spa (original task)
- `switch` — per-token switch label, 1 iff the word's language differs
  from the previous word's, else 0 (new target for Method C)


In [4]:
def build_sents(df_):
    out = []
    for (d, s), g in df_.groupby(["dialogue_id","sentence_id"], sort=False):
        g = g.sort_values("word_index")
        words = g["word"].tolist()
        y     = g["y"].astype(int).tolist()
        out.append({
            "did": int(d), "sid": int(s),
            "words_id": [enc_w(w) for w in words],
            "chars":    [enc_chars(w) for w in words],
            "pos":      [enc_p(p) for p in g["pos_tag"]],
            "y":        y,
            "switch":   [0] + [int(y[i] != y[i-1]) for i in range(1, len(y))],
        })
    return out

tr_sents = build_sents(data[data["split"]=="train"])
va_sents = build_sents(data[data["split"]=="val"])
te_sents = build_sents(data[data["split"]=="test"])
print(f"Sentences: train {len(tr_sents)}  val {len(va_sents)}  test {len(te_sents)}")

Sentences: train 32300  val 4628  test 6399


In [18]:
# Sanity check: look at one sentence
import pprint
ex = tr_sents[0]
print("words:  ", [list(word2id.keys())[i] for i in ex["words_id"]])
print("y:      ", ex["y"])
print("switch: ", ex["switch"])
print("chars[0]:", ex["chars"][0])

words:   ['cuándo', 'vamos', 'a', 'salir']
y:       [1, 1, 1, 1]
switch:  [0, 0, 0, 0]
chars[0]: [16, 12, 29, 6, 13, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 4. DataLoader — padded batches with char dimensions

char tensor is
3-D `(batch, sentence_len, MAX_CHAR)`. We allocate a zero tensor of that
shape and fill in each sentence.


In [6]:
class SentDS(Dataset):
    def __init__(self, sents, target_key):
        self.sents = sents; self.target_key = target_key
    def __len__(self): return len(self.sents)
    def __getitem__(self, i):
        s = self.sents[i]
        return (torch.tensor(s["words_id"],      dtype=torch.long),
                torch.tensor(s["pos"],           dtype=torch.long),
                torch.tensor(s["chars"],         dtype=torch.long),   # (L, MAX_CHAR)
                torch.tensor(s[self.target_key], dtype=torch.float))

def collate(batch):
    W  = pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0)
    P  = pad_sequence([b[1] for b in batch], batch_first=True, padding_value=0)
    Lmax = max(b[2].shape[0] for b in batch)
    Cpad = torch.zeros(len(batch), Lmax, MAX_CHAR, dtype=torch.long)
    for i, b in enumerate(batch):
        Cpad[i, :b[2].shape[0]] = b[2]
    Y = pad_sequence([b[3] for b in batch], batch_first=True, padding_value=0.0)
    M = (W != 0).float()                                 # 1 at real tokens, 0 at padding
    return W, P, Cpad, Y, M

BATCH = 32
def loaders(target_key):
    return (
        DataLoader(SentDS(tr_sents, target_key), BATCH, shuffle=True,  collate_fn=collate),
        DataLoader(SentDS(va_sents, target_key), BATCH, shuffle=False, collate_fn=collate),
        DataLoader(SentDS(te_sents, target_key), BATCH, shuffle=False, collate_fn=collate),
    )

## 5. The CharCNN sub-encoder

For each word we:

1. **Embed characters** into a small dense space (16-dim vectors).
2. Run a handful of **1-D convolutions** over the character sequence —
   think of them as sliding n-gram detectors of widths 2, 3, 4
   (bigrams, trigrams, 4-grams of characters).
3. **Max-pool** each convolution over the character axis — "did this
   n-gram appear anywhere in the word?"
4. Concatenate across kernel sizes and project down to a compact
   32-dim word feature.

Intuition: this lets the model learn features like "ends in `-ción`",
"contains `ñ`", "has an `-ing` suffix" from data. Those are strong
language cues even for rare or unseen words.

```
         char ids  (B, L, C)
             │
        embed to D            D = 16  (char_dim)
             │
        3 parallel Conv1D      kernel sizes (2, 3, 4), each F filters
             │
        max-pool over C        → 3 × F features per word
             │
        linear projection      → out_dim = 32
             ↓
        word char-feature  (B, L, 32)
```


In [7]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim=16, out_dim=32,
                 kernel_sizes=(2, 3, 4), filters=16):
        super().__init__()
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, filters, k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.proj = nn.Linear(filters * len(kernel_sizes), out_dim)

    def forward(self, char_ids):
        # char_ids: (B, L, C)
        B, L, C = char_ids.shape
        x = self.char_emb(char_ids)              # (B, L, C, D)
        x = x.view(B*L, C, -1).transpose(1, 2)   # (B*L, D, C)  — conv1d expects channels-first
        outs = [torch.relu(conv(x)).max(dim=-1).values for conv in self.convs]
        x = torch.cat(outs, dim=-1)              # (B*L, filters * n_kernels)
        x = self.proj(x)                         # (B*L, out_dim)
        return x.view(B, L, -1)

## 6. Tagger wrapper — word + POS (+ optional char) → BiLSTM → head

Same shape as the tutorial's BiLSTM, with an optional char feature
concatenated in.


In [8]:
class Tagger(nn.Module):
    def __init__(self, n_words, n_pos, n_chars=None,
                 wd=64, pd_=16, char_out=32,
                 use_char=False, hidden=64, dropout=0.3):
        super().__init__()
        self.use_char = use_char
        self.we = nn.Embedding(n_words, wd, padding_idx=0)
        self.pe = nn.Embedding(n_pos,   pd_, padding_idx=0)
        in_dim = wd + pd_
        if use_char:
            assert n_chars is not None
            self.char = CharCNN(n_chars, out_dim=char_out)
            in_dim += char_out
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden, 1)

    def forward(self, w, p, c):
        x = [self.we(w), self.pe(p)]
        if self.use_char:
            x.append(self.char(c))
        x = torch.cat(x, dim=-1)
        h, _ = self.lstm(x)
        return self.head(self.drop(h)).squeeze(-1)   # (B, L) logits

## 7. Shared training + prediction helpers

`pos_weight` up-weights the positive (rare) class in
the loss. doesn't just learn the
trivial "predict 0 everywhere" 

Concretely, with `pos_weight = (#neg / #pos) ≈ 55`, each true switch
contributes ~55× the loss of a non-switch. That tilts the model toward
finding switches at the cost of more false positives — exactly the
trade-off we'd want tuned by a threshold.


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_predict(model, target_key, epochs=3, pos_weight=None, lr=1e-3):
    tr_loader, va_loader, te_loader = loaders(target_key)
    pw = None if pos_weight is None else torch.tensor([pos_weight])
    crit = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pw)
    opt  = torch.optim.Adam(model.parameters(), lr=lr)

    def run_epoch(loader, train_mode):
        model.train(train_mode)
        total_loss = 0.0; total_tok = 0.0
        all_p, all_y = [], []
        for W, P, Cc, Y, M in loader:
            W, P, Cc, Y, M = W.to(device), P.to(device), Cc.to(device), Y.to(device), M.to(device)
            logits = model(W, P, Cc)
            lt = crit(logits, Y) * M
            loss = lt.sum() / M.sum().clamp(min=1)
            if train_mode:
                opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * M.sum().item()
            total_tok  += M.sum().item()
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            yy, mm = Y.cpu().numpy(), M.cpu().numpy()
            for b in range(probs.shape[0]):
                L = int(mm[b].sum())
                all_p.append(probs[b, :L]); all_y.append(yy[b, :L])
        return total_loss/max(total_tok, 1), np.concatenate(all_p), np.concatenate(all_y)

    for ep in range(1, epochs + 1):
        tl, tp, ty = run_epoch(tr_loader, True)
        vl, vp, vy = run_epoch(va_loader, False)
        try:
            tr_auc = roc_auc_score(ty, tp); va_auc = roc_auc_score(vy, vp)
        except ValueError:
            tr_auc = va_auc = float("nan")
        print(f"  ep {ep}  train loss {tl:.4f} auc {tr_auc:.3f} | "
              f"val loss {vl:.4f} auc {va_auc:.3f}")

    # --- collect per-token predictions on a given split, aligned to `data` ---
    def predict_all(sents):
        probs = np.full(len(data), np.nan)
        preds = np.full(len(data), np.nan)
        model.eval()
        with torch.no_grad():
            for s in sents:
                W  = torch.tensor([s["words_id"]], dtype=torch.long).to(device)
                P  = torch.tensor([s["pos"]],      dtype=torch.long).to(device)
                Cc = torch.tensor([s["chars"]],    dtype=torch.long).to(device)
                pr = torch.sigmoid(model(W, P, Cc)).cpu().numpy()[0]
                mask = ((data["dialogue_id"]==s["did"])
                        & (data["sentence_id"]==s["sid"]))
                idxs = sorted(data.index[mask].tolist(),
                              key=lambda i: data.at[i, "word_index"])
                for ii, pp in zip(idxs, pr):
                    probs[ii] = pp
                    preds[ii] = int(pp >= 0.5)
        return probs, preds

    val_probs, val_preds = predict_all(va_sents)
    te_probs,  te_preds  = predict_all(te_sents)
    return {"val_probs": val_probs, "val_preds": val_preds,
            "te_probs":  te_probs,  "te_preds":  te_preds}

In [10]:
# Evaluation helpers (same split-handling as switch_point_evaluation.ipynb).
def eval_token_language(out):
    m = (data["split"]=="test") & (~np.isnan(out["te_preds"]))
    y_t = data.loc[m, "y"].astype(int).values
    y_h = out["te_preds"][m.values].astype(int)
    return {"n": int(m.sum()),
            "acc": accuracy_score(y_t, y_h),
            "f1":  f1_score(y_t, y_h)}

def eval_switch_from_language(out):
    probs = pd.Series(out["te_probs"], index=data.index)
    preds = pd.Series(out["te_preds"], index=data.index)
    prev_pred = data.groupby(["dialogue_id","sentence_id"], sort=False)["y"].shift(1)  # placeholder
    # Compute prev_pred on the model's own predictions:
    prev_pred = preds.groupby([data["dialogue_id"], data["sentence_id"]], sort=False).shift(1)
    prev_prob = probs.groupby([data["dialogue_id"], data["sentence_id"]], sort=False).shift(1)
    sw_pred  = (prev_pred.notna() & (preds != prev_pred)).astype("float")
    sw_score = (probs - prev_prob).abs()
    m = ((data["split"]=="test") & data["prev_y"].notna()
         & preds.notna() & prev_pred.notna())
    yt = data.loc[m, "switch_true"].astype(int).values
    yp = sw_pred.loc[m].astype(int).values
    ss = sw_score.loc[m].values
    P, R, Fsc, _ = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)
    return {"n": int(m.sum()),
            "precision": P, "recall": R, "f1": Fsc,
            "auc": roc_auc_score(yt, ss),
            "ap":  average_precision_score(yt, ss)}

def eval_switch_direct(out, threshold=0.5):
    probs = pd.Series(out["te_probs"], index=data.index)
    preds = (probs >= threshold).astype("float")
    preds[probs.isna()] = np.nan
    m = (data["split"]=="test") & data["prev_y"].notna() & preds.notna()
    yt = data.loc[m, "switch_true"].astype(int).values
    yp = preds.loc[m].astype(int).values
    ss = probs.loc[m].values
    P, R, Fsc, _ = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)
    return {"n": int(m.sum()), "threshold": float(threshold),
            "precision": P, "recall": R, "f1": Fsc,
            "auc": roc_auc_score(yt, ss),
            "ap":  average_precision_score(yt, ss)}

## 8. Method A — reference baseline (word + POS, language target)

Runs the same model as the tutorial for a controlled comparison. Every
later variant is an ablation relative to this.


In [11]:
torch.manual_seed(SEED)
m_A = Tagger(len(word2id), len(pos2id), use_char=False)
print(f"  params: {sum(p.numel() for p in m_A.parameters()):,}")
out_A = train_and_predict(m_A, "y", epochs=3)

  params: 433,985
  ep 1  train loss 0.1899 auc 0.965 | val loss 0.0591 auc 0.998
  ep 2  train loss 0.1326 auc 0.976 | val loss 0.0776 auc 0.998
  ep 3  train loss 0.1136 auc 0.983 | val loss 0.0490 auc 0.998


In [12]:
tok_A = eval_token_language(out_A)
sw_A  = eval_switch_from_language(out_A)
print("Token-level:", {k: round(v, 3) if isinstance(v, float) else v for k, v in tok_A.items()})
print("Switch:     ", {k: round(v, 3) if isinstance(v, float) else v for k, v in sw_A.items()})

Token-level: {'n': 37027, 'acc': 0.986, 'f1': 0.987}
Switch:      {'n': 30628, 'precision': 0.675, 'recall': 0.554, 'f1': 0.609, 'auc': 0.966, 'ap': 0.693}


## 9. Method B — add char features (language target)

Same model, plus the CharCNN. This adds about 21k parameters (small — the
char vocab is only 38 characters) but gives the LSTM direct access to
spelling patterns at every token.


In [13]:
torch.manual_seed(SEED)
m_B = Tagger(len(word2id), len(pos2id), n_chars=len(char2id), use_char=True)
print(f"  params: {sum(p.numel() for p in m_B.parameters()):,}")
out_B = train_and_predict(m_B, "y", epochs=3)

  params: 454,897
  ep 1  train loss 0.1728 auc 0.966 | val loss 0.0490 auc 0.999
  ep 2  train loss 0.1228 auc 0.975 | val loss 0.0385 auc 0.999
  ep 3  train loss 0.1085 auc 0.982 | val loss 0.0375 auc 0.999


In [14]:
tok_B = eval_token_language(out_B)
sw_B  = eval_switch_from_language(out_B)
print("Token-level:", {k: round(v, 3) if isinstance(v, float) else v for k, v in tok_B.items()})
print("Switch:     ", {k: round(v, 3) if isinstance(v, float) else v for k, v in sw_B.items()})

Token-level: {'n': 37027, 'acc': 0.992, 'f1': 0.992}
Switch:      {'n': 30628, 'precision': 0.792, 'recall': 0.77, 'f1': 0.781, 'auc': 0.988, 'ap': 0.849}


**Reading the numbers:**

| metric       | A (no chars) | B (char CNN) | Δ |
|--------------|-------------:|-------------:|--:|
| Token F1     | 0.987 | 0.992 | +0.005 |
| **Switch F1**| **0.610** | **0.779** | **+0.169** |
| Switch AP    | 0.694 | 0.842 | +0.148 |

Adding char features barely moves the token-level F1 (it was already near
ceiling) but lifts **switch F1 by 17 points**. That's the largest
single-step improvement we've seen on this task — strong evidence that
what the BiLSTM was missing was spelling-level information about each word.

This is the concrete version of the "word identity matters" intuition
from section 9 of the main tutorial.


## 10. Method C — train directly on the switch target

Switch to the *new* target variable. Every token now gets label 1 if it's
a switch, 0 otherwise. Class balance is ~55:1 against switches, so we
pass `pos_weight` to `BCEWithLogitsLoss` to compensate.

Two things to watch:

1. **Train-time AUC drops** relative to Method B — not because the model
   is worse, but because switch detection is harder. Token-language AUC
   was 0.99 because 98 % of tokens were easy; switch AUC measures the
   harder 1.8 % directly.
2. At threshold 0.5, this model predicts **too many** switches (recall
   shoots up, precision drops) because `pos_weight` has shifted its
   logit distribution. We fix that with a threshold sweep on the val
   split in section 11.


In [15]:
# Recompute pos_weight from the train split.
train_switch_rate = np.mean([
    t for sent in tr_sents for t in sent["switch"][1:]   # skip first-word zero
])
pos_weight = (1 - train_switch_rate) / max(train_switch_rate, 1e-6)
print(f"train switch rate: {train_switch_rate:.4f}  ->  pos_weight {pos_weight:.1f}")

torch.manual_seed(SEED)
m_C = Tagger(len(word2id), len(pos2id), n_chars=len(char2id), use_char=True)
print(f"  params: {sum(p.numel() for p in m_C.parameters()):,}")
out_C = train_and_predict(m_C, "switch", epochs=3, pos_weight=pos_weight)

train switch rate: 0.0179  ->  pos_weight 54.9
  params: 454,897
  ep 1  train loss 0.9464 auc 0.810 | val loss 0.5914 auc 0.945
  ep 2  train loss 0.4001 auc 0.969 | val loss 0.2697 auc 0.987
  ep 3  train loss 0.2214 auc 0.990 | val loss 0.2342 auc 0.989


In [16]:
sw_C_05 = eval_switch_direct(out_C, threshold=0.5)
print("Method C @ thr=0.5:",
      {k: round(v, 3) if isinstance(v, float) else v for k, v in sw_C_05.items()})

Method C @ thr=0.5: {'n': 30628, 'threshold': 0.5, 'precision': 0.252, 'recall': 0.916, 'f1': 0.395, 'auc': 0.98, 'ap': 0.679}


The model now catches **93 % of real switches** — but 76 % of its
switch predictions are wrong (precision 0.24). At the default threshold
this looks like a regression, but notice:

- **Switch AUC = 0.984** — almost identical to Method B. The *ranking*
  is fine; only the decision threshold is wrong.
- Recall 0.93 means this model would be a great first stage in a
  two-stage pipeline, or useful in an application where missing a switch
  is costlier than a false alarm.


## 11. Threshold tuning for Method C

Because we trained with a positive class weight, the cutoff that
optimizes F1 isn't 0.5. We do a simple sweep on the **val** split and
apply the best threshold to test. This never sees test labels while
tuning, so it doesn't leak.


In [17]:
val_probs = pd.Series(out_C["val_probs"], index=data.index)
m_val = (data["split"]=="val") & data["prev_y"].notna() & val_probs.notna()
yv = data.loc[m_val, "switch_true"].astype(int).values
pv = val_probs.loc[m_val].values

best_thr, best_f1 = 0.5, 0.0
for thr in np.linspace(0.05, 0.99, 95):
    yp = (pv >= thr).astype(int)
    P, R, Fsc, _ = precision_recall_fscore_support(yv, yp, average="binary", zero_division=0)
    if Fsc > best_f1:
        best_f1, best_thr = Fsc, thr
print(f"Best val F1 = {best_f1:.3f}  at threshold = {best_thr:.2f}")

sw_C_tuned = eval_switch_direct(out_C, threshold=best_thr)
print("Method C (tuned) on test:",
      {k: round(v, 3) if isinstance(v, float) else v for k, v in sw_C_tuned.items()})

Best val F1 = 0.713  at threshold = 0.93
Method C (tuned) on test: {'n': 30628, 'threshold': 0.93, 'precision': 0.612, 'recall': 0.695, 'f1': 0.651, 'auc': 0.98, 'ap': 0.679}


Even with the threshold tuned, Method C (F1 ≈ 0.67) does **not** beat
Method B (F1 ≈ 0.78). 


## 12. Summary

| model                                  | tok acc | tok F1 |  sw P  |  sw R  | sw F1  | sw AUC | sw AP |
|----------------------------------------|--------:|-------:|-------:|-------:|-------:|-------:|------:|
| A · word+POS                           | 0.987   | 0.987  | 0.674  | 0.557  | **0.610** | 0.966 | 0.694 |
| B · word+POS+**char**                  | 0.992   | 0.992  | 0.797  | 0.761  | **0.779** | 0.988 | 0.842 |
| C · (B) + switch target, thr=0.5       |   –     |   –    | 0.236  | 0.932  | 0.376  | 0.984 | 0.697 |
| C · (B) + switch target, thr=0.95 (tuned) |   –  |   –    | 0.643  | 0.692  | 0.667  | 0.984 | 0.697 |

**Conclusion:** adding character features (A → B, switch F1
+0.17). Directly training on the switch target (C) doesn't help on
F1.

### What to try next

1. **Pretrained multilingual embeddings.** fastText's `cc.bi` vectors or
   an XLM-R encoder would replace the `nn.Embedding` lookup with
   language-aware contextual vectors. This is the logical next jump.
2. **Character BiLSTM instead of char CNN.** LSTMs over characters pick
   up longer morphological patterns (suffixes like `-mente`, `-ción`).
3. **Focal loss** on the switch target. Cleaner than pos_weight for
   heavily imbalanced binary classification.
4. **Ensemble B and C.** Use C's switch score as an additional feature
   for B's calibrated predictions, or average the two switch-detection
   scores.
5. **Per-switch analysis.** Bucket errors by POS of the switch point —
   we already know content words switch more; we should check where
   the remaining misses are concentrated.
